# Phase 1 — Agent Interfaces

**Dev 2 | Detection workstream**  
Validates the `BaseAgent` contract, data contracts, config integration, and
serialisability. Production logic lives in `src/agents/`; tests live in
`tests/test_agents.py`. This notebook imports and demonstrates — it does not
re-implement.

| Section | Purpose |
|---|---|
| 1 | Setup |
| 2 | Data contracts (`AlertScore` output schema) |
| 3 | `BaseAgent` contract walkthrough |
| 4 | `StubAgent` — minimal compliant implementation |
| 5 | Config & Nepal-context integration |
| 6 | Serialisation smoke test |
| 7 | Contract compliance summary |


---
## 1. Setup

In [ ]:
import sys, logging, random
from pathlib import Path
import numpy as np

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = Path('/kaggle/working/neural-sentinel')
assert (PROJECT_ROOT / 'src').exists(), f'src/ not found under {PROJECT_ROOT}'
sys.path.insert(0, str(PROJECT_ROOT))

random.seed(42); np.random.seed(42)
logging.basicConfig(level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(name)s | %(message)s',
    datefmt='%H:%M:%S')
print(f'PROJECT_ROOT: {PROJECT_ROOT}')


In [ ]:
# GPU/TPU detection — not required for Phase 1, pattern established for later phases
import os
DEVICE = 'cpu'
try:
    import torch as _t
    if _t.cuda.is_available():  DEVICE = 'cuda'
    elif getattr(getattr(_t, 'backends', None), 'mps', None) and _t.backends.mps.is_available():
        DEVICE = 'mps'
except ImportError: pass
if os.environ.get('TPU_NAME'): DEVICE = 'tpu'
print(f'Device: {DEVICE} (CPU expected for Phase 1)')


---
## 2. Data Contracts

Agents **read** `Transaction` rows and **write** `AlertScore` rows.
Both models are defined in `src/data_contracts.py` — the single source of truth
for all inter-agent data (AGENTS.md §5).

The `AlertScore` fields map directly to `BaseAgent.prediction_columns`:
`transaction_id`, `agent_name`, `risk_score`, `alert_flag`, `reason_code`,
`explanation`, `timestamp`.

In [ ]:
import datetime, pprint
from src.data_contracts import Transaction, AlertScore

# Confirm AlertScore fields match the agent output contract
print('AlertScore fields:')
for name, field in AlertScore.model_fields.items():
    print(f'  {name:<20} {str(field.annotation)}')


In [ ]:
# Build a representative AlertScore to confirm schema accepts agent output
alert = AlertScore(
    transaction_id='txn-001',
    agent_name='geo_risk',
    risk_score=0.82,
    alert_flag=1,
    reason_code='HIGH_RISK_CORRIDOR',
    explanation='NPR 450,000 remittance via Qatar->Nepal at 02:15; VPN detected.',
    timestamp=datetime.datetime.now(datetime.timezone.utc),
)
pprint.pprint(alert.model_dump())


In [ ]:
# Cross-border without corridor raises at parse time — data never reaches agents
from pydantic import ValidationError
try:
    Transaction(
        transaction_id='bad', transaction_date=datetime.date(2025,1,1),
        transaction_time=datetime.time(0,0), sender_account_id='A', receiver_account_id='B',
        transaction_type='transfer', amount_npr=100.0, original_currency='USD',
        exchange_rate=120.0, channel='online_banking', is_cross_border=1,
        remittance_corridor=None,  # invalid: cross-border must declare corridor
        device_type='desktop', ip_address='1.2.3.4', ip_is_vpn=0,
        is_fraud=0, aml_risk_indicator=0,
    )
except ValidationError as e:
    print('Caught expected ValidationError:', e.errors()[0]['msg'])


---
## 3. `BaseAgent` Contract

AGENTS.md §8.1 lists nine rules every agent must follow. `BaseAgent` enforces them
structurally:

| Rule | Mechanism |
|---|---|
| Inherit `BaseAgent` | `ABC` — instantiation fails without all abstract methods |
| Accept `config` + `logger` | `__init__` signature + `_config_to_dict` |
| `fit` / `predict` / `explain` | `@abstractmethod` |
| Canonical output schema | `build_predictions` enforces `prediction_columns` |
| Score range `[0, 1]` | `build_predictions` clips and logs on violation |
| Serialisable | `__getstate__` / `__setstate__` strip live logger |
| No mutation of input | `build_predictions` copies `transaction_id` only |
| Missing features graceful | `require_columns` returns `False` → caller returns `empty_predictions()` |
| Alert threshold resolution | agent-specific key → generic key → class default (0.5) |


In [ ]:
import inspect
from src.agents.base_agent import BaseAgent

print('prediction_columns:')
for col in BaseAgent.prediction_columns:
    print(f'  {col}')

print('\nAbstract methods (must be implemented):')
for name in sorted(m for m, v in inspect.getmembers(BaseAgent)
                   if getattr(v, '__isabstractmethod__', False)):
    print(f'  {name}{inspect.signature(getattr(BaseAgent, name))}')


In [ ]:
import pandas as pd

# Threshold resolution: agent-specific key takes priority
class _VAgent(BaseAgent):
    agent_name = 'velocity'
    def fit(self, d): return self
    def predict(self, d): return self.empty_predictions()
    def explain(self, t): return ''

cases = [
    ('agent-specific key', {'velocity_alert_threshold': 0.7}),
    ('generic key',        {'alert_threshold': 0.3}),
    ('empty config',       {}),
]
for label, cfg in cases:
    print(f'{label:<25} → threshold = {_VAgent(config=cfg).alert_threshold}')


In [ ]:
# build_predictions: clips scores, sets alert_flag, stamps timestamp
class _Demo(BaseAgent):
    agent_name = 'demo'
    def fit(self, d): return self
    def predict(self, d): return self.build_predictions(d, [1.8])  # out-of-range → clipped
    def explain(self, t): return ''

row = pd.DataFrame({'transaction_id': ['t1']})
out = _Demo().fit(row).predict(row)
print('Clipped score:', out.risk_score.iloc[0])  # must be 1.0
print('Columns match prediction_columns:', list(out.columns) == list(BaseAgent.prediction_columns))


In [ ]:
# empty_predictions: schema-safe empty DataFrame for graceful failure paths
empty = BaseAgent.empty_predictions()
print('empty_predictions() is empty:', empty.empty)
print('Schema preserved:', list(empty.columns) == list(BaseAgent.prediction_columns))


---
## 4. `StubAgent` — Minimal Compliant Implementation

`StubAgent` is the canonical template for new agent authors. It does the absolute
minimum to satisfy the contract: sets `is_fitted`, calls `build_predictions` with
a fixed score, returns an f-string explanation. Everything else is inherited.

The same class is used in `tests/test_agents.py`.

In [ ]:
from src.agents.base_agent import BaseAgent
import pandas as pd


class StubAgent(BaseAgent):
    """Minimal BaseAgent — fixed 0.2 risk score, no model training."""

    agent_name = 'stub'

    def fit(self, data: pd.DataFrame) -> 'StubAgent':
        self.is_fitted = True
        self.logger.info('StubAgent fitted on %d rows', len(data))
        return self

    def predict(self, data: pd.DataFrame) -> pd.DataFrame:
        if not self.require_columns(data, ('transaction_id',)):
            return self.empty_predictions()
        return self.build_predictions(
            data,
            risk_scores=[0.2] * len(data),
            reason_code='STUB_SCORE',
        )

    def explain(self, transaction_id: str) -> str:
        return f'Transaction {transaction_id}: stub baseline score 0.2.'


In [ ]:
txns = pd.DataFrame({
    'transaction_id': ['t1', 't2', 't3'],
    'amount_npr': [50_000.0, 1_200_000.0, 8_500.0],
})

stub = StubAgent(config={'stub_alert_threshold': 0.15}).fit(txns)
preds = stub.predict(txns)
print(preds.to_string(index=False))


In [ ]:
# Missing transaction_id — must return empty schema, not raise
bad = pd.DataFrame({'amount_npr': [100.0]})
result = stub.predict(bad)
print('Empty on bad input:', result.empty,
      '| Schema intact:', list(result.columns) == list(BaseAgent.prediction_columns))

# Input immutability
before = txns.copy(deep=True)
stub.predict(txns)
print('Input mutated:', not txns.equals(before))  # must be False

# explain
print(stub.explain('t1'))


---
## 5. Config & Nepal-Context Integration

AGENTS.md §10.7: all magic numbers live in `src/utils/config.py` and
`src/utils/nepal_context.py`. Agents import constants — they never hardcode values.

In [ ]:
from src.utils.config import get_config
from src.utils.nepal_context import (
    REMITTANCE_CORRIDORS, CORRIDOR_RISK_SCORES,
    NRB_CASH_REPORTING_THRESHOLD_NPR, CHANNEL_MIX,
)

cfg = get_config()

print('=== Per-agent alert thresholds ===')
for field in ['velocity_alert_threshold', 'geo_risk_alert_threshold',
              'kyc_aml_alert_threshold', 'meta_learner_alert_threshold']:
    print(f'  {field:<40} {getattr(cfg, field)}')

print(f'\n=== NRB cash reporting threshold ===\n  NPR {NRB_CASH_REPORTING_THRESHOLD_NPR:,.0f}')


In [ ]:
print('=== Remittance corridors (sample) ===')
for corridor, risk_tier in list(REMITTANCE_CORRIDORS.items())[:5]:
    score = CORRIDOR_RISK_SCORES[risk_tier]
    print(f'  {corridor:<30} risk={risk_tier:<8} score={score:.2f}')

print('\n=== Banking channel mix ===')
for ch, w in sorted(CHANNEL_MIX.items(), key=lambda x: -x[1]):
    print(f'  {ch:<20} {w*100:.1f}%')


In [ ]:
# Agent consumes config
cfg_dict = cfg.model_dump()
agent = StubAgent(config=cfg_dict)
print(f'StubAgent from Config singleton: threshold={agent.alert_threshold}')


---
## 6. Serialisation

AGENTS.md §8.1 rule 6: agents must be serialisable so fitted models can be saved and
loaded without refitting. `BaseAgent.__getstate__` strips the live logger; 
`__setstate__` re-attaches it by name.

In [ ]:
import pickle, joblib, tempfile
from pathlib import Path

stub = StubAgent(config={'alert_threshold': 0.35}).fit(txns)

# pickle
restored = pickle.loads(pickle.dumps(stub))
print('pickle  — type:', type(restored).__name__,
      '| threshold:', restored.alert_threshold,
      '| fitted:', restored.is_fitted,
      '| logger:', restored.logger.name)

# joblib
with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / 'stub.joblib'
    joblib.dump(stub, path)
    restored_jl = joblib.load(path)
print('joblib  — type:', type(restored_jl).__name__,
      '| predict works:', len(restored_jl.predict(txns)) == len(txns))


---
## 7. Contract Compliance Summary

Quick automated check of the nine AGENTS.md §8.1 rules against `StubAgent`.
Full test coverage is in `tests/test_agents.py`.

In [ ]:
import inspect, logging as _log

def check(AgentClass, txns):
    r = {}
    sig = inspect.signature(AgentClass.__init__).parameters
    r['R1 inherits BaseAgent']       = issubclass(AgentClass, BaseAgent)
    r['R2 config+logger params']     = 'config' in sig and 'logger' in sig
    a = AgentClass(config={}, logger=_log.getLogger('check'))
    r['R3 fit returns self']         = a.fit(txns) is a
    p = a.predict(txns)
    r['R4 predict required columns'] = {'transaction_id','risk_score','alert_flag','reason_code'}.issubset(p.columns)
    r['R5 explain returns str']      = isinstance(a.explain('t1'), str)
    try:
        import pickle as _pk
        r['R6 pickle serialisable']  = isinstance(_pk.loads(_pk.dumps(a)), AgentClass)
    except Exception as e:
        r['R6 pickle serialisable']  = str(e)
    r['R7 no print() in source']     = 'print(' not in inspect.getsource(AgentClass)
    before = txns.copy(deep=True); a.predict(txns)
    r['R8 input not mutated']        = txns.equals(before)
    try:
        res = a.predict(pd.DataFrame({'transaction_id': ['x']}))
        r['R9 missing features OK']  = isinstance(res, pd.DataFrame)
    except Exception as e:
        r['R9 missing features OK']  = str(e)
    return r

results = check(StubAgent, txns)
all_pass = True
for rule, outcome in results.items():
    ok = outcome is True
    if not ok: all_pass = False
    print(f"  [{'PASS' if ok else 'FAIL'}] {rule}")
print(f'\nOverall: {"ALL PASS" if all_pass else "ISSUES DETECTED"}')


---
## Conclusion

`BaseAgent` contract is locked. `StubAgent` passes all nine compliance rules.
Phase 2 builds `VelocityAgent` and `GeoRiskAgent` on top of this foundation —
see `notebooks/detection/phase2_velocity_geo_agents.ipynb`.

Run `pytest tests/test_agents.py -v` for the full test suite.